In [5]:
import ee

print("Earth Engine version:", ee.__version__)

Earth Engine version: 1.7.42


In [6]:
ee.Authenticate()

True

In [7]:
ee.Authenticate(force=True)

Enter verification code:  4/1ATsMZqBsorSqA3SsMHex57qBQE_7DFhLQnBxQEmvZDJS6nKdJ9kIVjsZRx0


EEException: Cannot authenticate: Invalid request.

In [8]:
import ee

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized successfully!")

✅ Earth Engine initialized successfully!


In [9]:
import ee
import pandas as pd
import os

# =========================================================
# 1. Initialize Google Earth Engine
# =========================================================

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized")


# =========================================================
# 2. Joshimath location
# =========================================================

joshimath = ee.Geometry.Point([
    79.56436,   # longitude
    30.55543    # latitude
])

# 10 km buffer
roi = joshimath.buffer(10000)

print("✅ Joshimath 10 km study area created")


# =========================================================
# 3. NASA Global Landslide Catalog
# =========================================================

landslides = ee.FeatureCollection(
    "projects/sat-io/open-datasets/events/global_landslide_1970-2019"
)

print("✅ Global Landslide Catalog loaded")


# =========================================================
# 4. Select landslides inside Joshimath area
# =========================================================

joshimath_landslides = landslides.filterBounds(roi)

print(
    "Total landslide records:",
    joshimath_landslides.size().getInfo()
)


# =========================================================
# 5. Output folder
# =========================================================

output_folder = "data/historical_landslide_monthly"

os.makedirs(
    output_folder,
    exist_ok=True
)


# =========================================================
# 6. Get all landslide records
# =========================================================

data = joshimath_landslides.getInfo()

print("✅ Landslide records downloaded to Python")


# =========================================================
# 7. Convert GEE features to Pandas
# =========================================================

rows = []

for feature in data["features"]:

    properties = feature.get("properties", {})
    geometry = feature.get("geometry", {})

    coordinates = geometry.get(
        "coordinates",
        [None, None]
    )

    longitude = coordinates[0]
    latitude = coordinates[1]


    # -----------------------------------------------------
    # Try to identify the event date
    # -----------------------------------------------------

    event_date = (
        properties.get("event_date")
        or properties.get("date")
        or properties.get("eventDate")
    )


    # -----------------------------------------------------
    # Store record
    # -----------------------------------------------------

    rows.append({

        "event_date":
            event_date,

        "latitude":
            latitude,

        "longitude":
            longitude,

        "landslide_type":
            properties.get("landslide_type"),

        "trigger":
            properties.get("trigger"),

        "source":
            properties.get("source"),

        "location":
            properties.get("location"),

        "country":
            properties.get("country"),

        "admin_division":
            properties.get("admin_division"),

        "fatalities":
            properties.get("fatalities"),

        "injuries":
            properties.get("injuries"),

        "year":
            properties.get("year"),

        "month":
            properties.get("month")
    })


# =========================================================
# 8. Create DataFrame
# =========================================================

df = pd.DataFrame(rows)

print(
    "Total Joshimath landslide records:",
    len(df)
)

print("\nAvailable columns:")
print(df.columns.tolist())


# =========================================================
# 9. Convert event date if available
# =========================================================

if "event_date" in df.columns:

    df["event_date"] = pd.to_datetime(
        df["event_date"],
        errors="coerce"
    )

    df["year"] = df["event_date"].dt.year

    df["month"] = df["event_date"].dt.month


# =========================================================
# 10. Remove records without a valid date
# =========================================================

df = df.dropna(
    subset=["year", "month"]
).copy()

df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)


# =========================================================
# 11. Create separate monthly CSV files
# =========================================================

if len(df) > 0:

    for year in sorted(df["year"].unique()):

        year_df = df[
            df["year"] == year
        ]

        for month in sorted(
            year_df["month"].unique()
        ):

            month_df = year_df[
                year_df["month"] == month
            ].copy()


            filename = (
                f"{output_folder}/"
                f"landslide_{year}_{month:02d}.csv"
            )


            month_df.to_csv(
                filename,
                index=False
            )


            print(
                f"✅ Saved: {filename} "
                f"({len(month_df)} events)"
            )


else:

    print(
        "\n⚠️ No dated landslide records "
        "were found inside the Joshimath 10-km area."
    )


print(
    "\n🎉 HISTORICAL LANDSLIDE MONTHLY DATA COMPLETED!"
)

✅ Earth Engine initialized
✅ Joshimath 10 km study area created
✅ Global Landslide Catalog loaded
Total landslide records: 6
✅ Landslide records downloaded to Python
Total Joshimath landslide records: 6

Available columns:
['event_date', 'latitude', 'longitude', 'landslide_type', 'trigger', 'source', 'location', 'country', 'admin_division', 'fatalities', 'injuries', 'year', 'month']
✅ Saved: data/historical_landslide_monthly/landslide_2007_07.csv (1 events)
✅ Saved: data/historical_landslide_monthly/landslide_2015_04.csv (2 events)
✅ Saved: data/historical_landslide_monthly/landslide_2016_09.csv (1 events)
✅ Saved: data/historical_landslide_monthly/landslide_2017_04.csv (1 events)
✅ Saved: data/historical_landslide_monthly/landslide_2017_05.csv (1 events)

🎉 HISTORICAL LANDSLIDE MONTHLY DATA COMPLETED!
